# Tratamiento de imagen · Semanas 8 y 9
## CADI Percepción Computacional · Experiencia IMAGEN

**Joan Schneider Beltrán Delgado** · Universidad de Cundinamarca

Diagnóstico de cinco condiciones de captura sobre la misma escena y corrección
de la imagen con baja iluminación.

**Origen de las imágenes.** Las condiciones 1 y 2 son capturas fotográficas
reales del mismo encuadre, tomadas del conjunto LOL (Wei et al., 2018), que se
construyó variando el tiempo de exposición y la ISO de la cámara. Las
condiciones 3, 4 y 5 se derivan por software de la imagen de referencia, porque
el conjunto no las incluye; el código que las genera está en este cuaderno.

## 1. Entorno

In [ ]:
!pip -q install opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)

## 2. Descarga de las dos capturas reales

Se descargan dos imágenes del conjunto LOL: la misma escena con exposición
normal y con poca luz.

In [ ]:
import urllib.request

BASE = ("https://raw.githubusercontent.com/JoanBeltranAlt/"
        "tratamiento-imagen-percepcion/main/imagenes/")

Path("imagenes").mkdir(exist_ok=True)
for n in ["01_referencia.png", "02_baja_luz.png", "03_desenfoque.png",
          "04_fondo_complejo.png", "05_angulo.png"]:
    destino = Path("imagenes") / n
    if not destino.exists():
        urllib.request.urlretrieve(BASE + n, destino)
    print(f"{n:24s} {destino.stat().st_size / 1024:6.0f} KB")

## 3. Cómo se derivaron las condiciones 3, 4 y 5

Este bloque documenta el procedimiento. No hace falta ejecutarlo, porque las
imágenes ya vienen descargadas, pero deja el método a la vista.

Para el desenfoque se usa una convolución con un núcleo lineal inclinado, que
reproduce la estela de una cámara movida. Se descarta `GaussianBlur` porque su
difuminado es isotrópico y no corresponde a ese fenómeno.

In [ ]:
def desenfoque_movimiento(img, largo=21, angulo=18.0):
    """Estela direccional, no un borrón uniforme."""
    k = np.zeros((largo, largo), np.float32)
    k[largo // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((largo / 2 - 0.5, largo / 2 - 0.5), angulo, 1)
    k = cv2.warpAffine(k, M, (largo, largo))
    return cv2.filter2D(img, -1, k / k.sum())


def otro_angulo(img, grados=-16.0, escala=0.74):
    """Rota y aleja la escena, como una toma desde otra posición."""
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), grados, escala)
    borde = tuple(int(v) for v in img[2:12, 2:12].reshape(-1, 3).mean(0))
    return cv2.warpAffine(img, M, (w, h), borderValue=borde)

## 4. Diagnóstico

Mide las características que pide el enunciado y añade tres indicadores:
entropía (información tonal), densidad de bordes (estructura detectable) y
ruido en la zona más uniforme, necesario para comprobar después si el
tratamiento lo amplificó.

In [ ]:
def tenengrad(gris):
    """Energía de gradiente de Sobel: segunda medida de nitidez."""
    gx = cv2.Sobel(gris, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gris, cv2.CV_64F, 0, 1, ksize=3)
    return float((gx ** 2 + gy ** 2).mean())


def ruido_zona_plana(gris, lado=64):
    """Desviación típica del parche más uniforme que se encuentre."""
    h, w = gris.shape
    mejor = None
    for y in range(0, h - lado, max(1, (h - lado) // 8)):
        for x in range(0, w - lado, max(1, (w - lado) // 8)):
            d = float(gris[y:y + lado, x:x + lado].std())
            if mejor is None or d < mejor:
                mejor = d
    return mejor if mejor is not None else float(gris.std())


def densidad_bordes(gris):
    """Porcentaje de píxeles que Canny marca como borde."""
    bordes = cv2.Canny(cv2.GaussianBlur(gris, (5, 5), 0), 80, 160)
    return float((bordes == 255).mean() * 100)


def medir(ruta):
    img = cv2.imread(str(ruta))
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    alto, ancho = img.shape[:2]

    hist = cv2.calcHist([gris], [0], None, [256], [0, 256]).ravel()
    p = hist / hist.sum()
    p = p[p > 0]
    p1, p99 = np.percentile(gris, [1, 99])

    return dict(
        archivo=ruta.name,
        resolucion=f"{ancho}x{alto}",
        canales=img.shape[2],
        brillo=round(float(gris.mean()), 1),
        contraste=round(float(gris.std()), 1),
        nitidez=round(float(cv2.Laplacian(gris, cv2.CV_64F).var()), 1),
        tenengrad=round(tenengrad(gris), 1),
        entropia=round(float(-(p * np.log2(p)).sum()), 2),
        amplitud_rango=int(p99 - p1),
        pct_subexpuesto=round(float((gris < 30).mean() * 100), 1),
        densidad_bordes=round(densidad_bordes(gris), 2),
        ruido_zona_plana=round(ruido_zona_plana(gris), 2),
        peso_kb=round(ruta.stat().st_size / 1024, 1),
        formato=ruta.suffix.lstrip(".").upper())

### Tabla comparativa

In [ ]:
CONDICIONES = [("01_referencia", "Referencia"),
               ("02_baja_luz", "Baja iluminación"),
               ("03_desenfoque", "Desenfoque o movimiento"),
               ("04_fondo_complejo", "Fondo complejo"),
               ("05_angulo", "Cambio de ángulo")]

filas = []
for base, desc in CONDICIONES:
    m = medir(Path("imagenes") / f"{base}.png")
    m["condicion"] = desc
    filas.append(m)

cols = ["condicion", "resolucion", "canales", "brillo", "contraste",
        "nitidez", "entropia", "pct_subexpuesto", "peso_kb", "formato"]
anchos = [max(len(c), max(len(str(f[c])) for f in filas)) for c in cols]
print("  ".join(c.upper().ljust(w) for c, w in zip(cols, anchos)))
for f in filas:
    print("  ".join(str(f[c]).ljust(w) for c, w in zip(cols, anchos)))

La toma de baja iluminación es la más degradada: cuatro de cada cinco
píxeles quedan por debajo del valor 30 y la densidad de bordes casi desaparece.

Conviene fijarse en el fondo complejo, que obtiene la **nitidez más alta** del
conjunto pese a ser la imagen más difícil de interpretar. La varianza del
laplaciano mide variación local, no calidad, y está contando el desorden del
fondo como si fuera detalle útil.

## 5. Visualización de las cinco condiciones

In [ ]:
plt.figure(figsize=(14, 7))
for i, (base, desc) in enumerate(CONDICIONES, 1):
    im = cv2.imread(str(Path("imagenes") / f"{base}.png"))
    plt.subplot(2, 3, i)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(f"{i}. {desc}", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Tratamiento de la imagen problemática

El diagnóstico señala subexposición, no falta de nitidez, así que el
tratamiento debe responder a eso.

Se descartaron dos alternativas. La ecualización global (`cv2.equalizeHist`)
estira todo el histograma y en una imagen tan oscura amplifica el ruido de las
sombras. La corrección gamma aclara de forma uniforme pero no separa los tonos
vecinos, que es lo que aquí falta.

CLAHE opera por regiones y limita cuánto amplifica cada una. Se aplica solo
sobre el canal L del espacio LAB, que codifica la luminancia: hacerlo sobre los
tres canales BGR desplazaría el color.

In [ ]:
def tratar_baja_luz(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l2 = clahe.apply(l)                       # solo la luminancia

    tratada = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)
    # realzar zonas oscuras amplifica el ruido del sensor
    return cv2.fastNlMeansDenoisingColored(tratada, None, 6, 6, 7, 21)


original = cv2.imread("imagenes/02_baja_luz.png")
tratada = tratar_baja_luz(original)

Path("resultados").mkdir(exist_ok=True)
cv2.imwrite("resultados/tratada_clahe.png", tratada)
print("tratamiento aplicado")

### Comparación numérica antes y después

In [ ]:
def perfil(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([g], [0], None, [256], [0, 256]).ravel()
    p = hist / hist.sum()
    p = p[p > 0]
    p1, p99 = np.percentile(g, [1, 99])
    return dict(brillo=round(float(g.mean()), 1),
                contraste=round(float(g.std()), 1),
                nitidez=round(float(cv2.Laplacian(g, cv2.CV_64F).var()), 1),
                tenengrad=round(tenengrad(g), 1),
                entropia=round(float(-(p * np.log2(p)).sum()), 2),
                amplitud_rango=int(p99 - p1),
                pct_subexpuesto=round(float((g < 30).mean() * 100), 1),
                densidad_bordes=round(densidad_bordes(g), 2),
                ruido_zona_plana=round(ruido_zona_plana(g), 2))


a, d = perfil(original), perfil(tratada)
r = perfil(cv2.imread("imagenes/01_referencia.png"))

print(f"{'MÉTRICA':22s}{'ANTES':>10s}{'DESPUÉS':>10s}"
      f"{'CAMBIO':>10s}{'REFERENCIA':>12s}")
for k in a:
    c = round(100 * (d[k] - a[k]) / a[k], 1) if a[k] else None
    txt = f"{c:+.1f}%" if c is not None else "—"
    print(f"{k:22s}{a[k]:>10}{d[k]:>10}{txt:>10}{r[k]:>12}")

Todas las métricas mejoran. La más relevante es la caída de píxeles
subexpuestos: más de la mitad de los que estaban aplastados contra el negro
recuperan un valor distinguible.

El ruido **baja** en lugar de subir, que es lo contrario de lo esperable al
realzar una imagen oscura. La razón está en el segundo paso: el filtro de
medias no locales suprime más ruido del que CLAHE introduce. Verificarlo
importa porque, sin esa comprobación, un aumento de nitidez podría estar
reflejando ruido amplificado y no detalle recuperado.

## 7. Comparación visual

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, (im, t) in zip(ax, [(original, "Antes · baja iluminación"),
                            (tratada, "Después · CLAHE + denoise"),
                            (cv2.imread("imagenes/01_referencia.png"),
                             "Referencia")]):
    a_.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    a_.set_title(t, fontsize=10)
    a_.axis("off")
plt.tight_layout()
plt.show()

### Histogramas

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2), sharey=True)
datos = [(original, "Antes", "#B4483C"),
         (tratada, "Después", "#1F7A52"),
         (cv2.imread("imagenes/01_referencia.png"), "Referencia", "#2F4FA8")]
for a_, (im, t, color) in zip(ax, datos):
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    a_.hist(g.ravel(), bins=64, range=(0, 255), color=color, alpha=.85)
    a_.set_title(f"{t} · media {g.mean():.1f} · σ {g.std():.1f}", fontsize=9)
    a_.set_xlim(0, 255)
plt.tight_layout()
plt.show()

## 8. Qué muestra el resultado

El tratamiento mejoró la imagen, pero no la igualó a la referencia. CLAHE
redistribuye la información que la imagen contiene; no inventa la que el sensor
nunca capturó. Donde la exposición fue tan corta que varios tonos quedaron
registrados con el mismo valor, no hay nada que separar.

**Sobre las métricas de evaluación.** PSNR y SSIM son métricas *full-reference*:
comparan dos imágenes píxel a píxel y exigen que estén alineadas. Aquí no
existe esa pareja, porque la imagen de referencia es otra toma con su propio
encuadre. Usarlas mediría la diferencia entre capturas, no el efecto del
tratamiento. Por eso la evaluación compara las mismas métricas antes y después,
contrastándolas con el rango que marca la referencia.